In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym


# ---------- Setup ----------
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)

N_STATES, N_ACTIONS = 16, 4
TERMINAL = {5, 7, 11, 12, 15}  # holes + goal


def one_hot(s):
    v = np.zeros(N_STATES, dtype=np.float32)
    v[s] = 1.0
    return v


# ---------- 1. Collect imperfect dataset ----------
def behavior_policy():
    if np.random.rand() < 0.3:
        return env.action_space.sample()  # 30% random

    return np.random.choice([1, 2])  # 70% Down/Right only


S, A, R, S2, D = [], [], [], [], []

for _ in range(300):
    s, _ = env.reset()
    done = False

    while not done:
        a = behavior_policy()

        s2, r, term, trunc, _ = env.step(a)

        done = term or trunc

        S.append(s)
        A.append(a)
        R.append(r)
        S2.append(s2)
        D.append(float(term))

        s = s2


S = np.array(S)
A = np.array(A)
S2 = np.array(S2)

R = np.array(R, dtype=np.float32)
D = np.array(D, dtype=np.float32)


# Split (state, action) pairs into common vs rare
pair_counts = np.zeros(
    (N_STATES, N_ACTIONS),
    dtype=int
)

for s, a in zip(S, A):
    pair_counts[s, a] += 1


valid = [
    s for s in range(N_STATES)
    if s not in TERMINAL
]

common = [
    (s, a)
    for s in valid
    for a in range(4)
    if pair_counts[s, a] >= 5
]

rare = [
    (s, a)
    for s in valid
    for a in range(4)
    if pair_counts[s, a] < 5
]


# ---------- 2. Q-Network ----------
class QNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(N_STATES, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, N_ACTIONS)
        )

    def forward(self, x):
        return self.net(x)


# ---------- 3. Loss: Bellman + alpha * CQL penalty ----------
def cql_loss(
    q_net,
    target_net,
    s,
    a,
    r,
    s2,
    done,
    gamma,
    alpha
):
    q_all = q_net(s)

    q_data = q_all.gather(
        1,
        a.unsqueeze(1)
    ).squeeze(1)

    with torch.no_grad():
        target = (
            r
            + gamma * (1 - done)
            * target_net(s2).max(dim=1).values
        )

    bellman = F.mse_loss(
        q_data,
        target
    )

    cql_pen = (
        torch.logsumexp(q_all, dim=1)
        - q_data
    ).mean()

    return bellman + alpha * cql_pen


# ---------- 4. Offline Training ----------
S_T = torch.tensor(
    np.array([one_hot(x) for x in S]),
    dtype=torch.float32
)

S2_T = torch.tensor(
    np.array([one_hot(x) for x in S2]),
    dtype=torch.float32
)

A_T = torch.tensor(
    A,
    dtype=torch.long
)

R_T = torch.tensor(
    R,
    dtype=torch.float32
)

D_T = torch.tensor(
    D,
    dtype=torch.float32
)


def train(
    alpha,
    steps=3000,
    batch=64,
    lr=1e-3,
    gamma=0.99
):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    q = QNet()
    tgt = QNet()

    tgt.load_state_dict(
        q.state_dict()
    )

    opt = torch.optim.Adam(
        q.parameters(),
        lr=lr
    )

    for step in range(steps):

        i = np.random.randint(
            0,
            len(S),
            batch
        )

        loss = cql_loss(
            q,
            tgt,
            S_T[i],
            A_T[i],
            R_T[i],
            S2_T[i],
            D_T[i],
            gamma,
            alpha
        )

        opt.zero_grad()

        loss.backward()

        opt.step()

        if step % 100 == 0:
            tgt.load_state_dict(
                q.state_dict()
            )

    return q


# ---------- 5. Helpers ----------
def q_table(q):
    with torch.no_grad():
        states = torch.tensor(
            np.array([
                one_hot(s)
                for s in range(N_STATES)
            ]),
            dtype=torch.float32
        )

        return q(states).numpy()


def evaluate(q):
    s, _ = env.reset()

    total = 0.0
    done = False
    steps = 0

    while not done and steps < 50:

        with torch.no_grad():
            state_tensor = torch.tensor(
                one_hot(s),
                dtype=torch.float32
            ).unsqueeze(0)

            a = q(
                state_tensor
            ).argmax().item()

        s, r, term, trunc, _ = env.step(a)

        done = term or trunc

        total += r
        steps += 1

    return total


def avg_q(qt, pairs):
    return float(
        np.mean([
            qt[s, a]
            for s, a in pairs
        ])
    )


# ---------- 6. Run and print key results ----------

# Train Offline DQN
dqn_model = train(alpha=0.0)

# Train CQL
cql_model = train(alpha=1.0)

# Get Q-tables
dqn_q = q_table(dqn_model)
cql_q = q_table(cql_model)

# Evaluate models
dqn_ret = evaluate(dqn_model)
cql_ret = evaluate(cql_model)


# Dataset statistics
print(
    f"Dataset: {len(S)} transitions | "
    f"Left used {100 * np.mean(A == 0):.0f}%, "
    f"Down {100 * np.mean(A == 1):.0f}%, "
    f"Right {100 * np.mean(A == 2):.0f}%, "
    f"Up {100 * np.mean(A == 3):.0f}%"
)

print(
    f"Rare/unseen (state,action) pairs: "
    f"{len(rare)} of "
    f"{len(rare) + len(common)}\n"
)


# Table header
print(
    f"{'':<12}"
    f"{'Return':>8}"
    f"{'Avg Q common':>15}"
    f"{'Avg Q rare':>13}"
    f"{'Rare Q > 1.0':>15}"
)


# Results
for name, qt, ret in [
    ("Offline DQN", dqn_q, dqn_ret),
    ("CQL", cql_q, cql_ret)
]:

    over = sum(
        qt[s, a] > 1.0
        for s, a in rare
    )

    print(
        f"{name:<12}"
        f"{ret:>8.1f}"
        f"{avg_q(qt, common):>15.3f}"
        f"{avg_q(qt, rare):>13.3f}"
        f"{over:>11d}/{len(rare)}"
    )


print(
    "\nNote: the max possible reward is 1.0, "
    "so any Q above 1.0 is overestimation."
)

Dataset: 1229 transitions | Left used 7%, Down 42%, Right 44%, Up 7%
Rare/unseen (state,action) pairs: 8 of 44

              Return   Avg Q common   Avg Q rare   Rare Q > 1.0
Offline DQN      0.0          0.812        0.756          6/8
CQL              0.0          1.238        0.571          1/8

Note: the max possible reward is 1.0, so any Q above 1.0 is overestimation.
